# Debris Mapper — train the detector
Runtime → Change runtime type → **T4 GPU**. Add `ROBOFLOW_API_KEY` under the 🔑 Secrets panel.

Raw data lives on Google Drive so it survives Colab restarts.

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')
import os
os.environ['ROBOFLOW_API_KEY'] = userdata.get('ROBOFLOW_API_KEY')

In [ ]:
!git clone -b ios-scaffold https://github.com/apandji/debris-detection.git
%cd debris-detection/training
!pip -q install -r requirements.txt

Keep `data/` on Drive. Put manual downloads (D-Fire, GDBDA, FEMA test set) in `MyDrive/debris-data/raw/<source id>/`.

In [ ]:
!mkdir -p /content/drive/MyDrive/debris-data/raw
!rm -rf data && mkdir -p data && ln -s /content/drive/MyDrive/debris-data/raw data/raw
!python scripts/fetch.py --list

## 1. Download Roboflow sources
`fetch.py` edits `status:` in this clone's sources.yaml — copy those changes back into the repo (or tell Claude).

In [ ]:
!python scripts/fetch.py

## 2. Eyeball each source
Open `data/sheets/<id>/index.html` from the Files panel (or download it). Mark good sources `status: checked`.

In [ ]:
!python scripts/sheet.py

## 3. Merge
Run `--check` first and fill any UNMAPPED classes in sources.yaml.

In [ ]:
!python scripts/merge.py --check --include-downloaded

In [ ]:
!python scripts/merge.py --include-downloaded --link
!cat data/merged/stats.md

## 4. Train
Start with a short run to prove the pipeline, then the full run.

In [ ]:
!python scripts/train.py --epochs 10 --name smoke-test

In [ ]:
# !python scripts/train.py --epochs 100 --name v0 --test

## 5. Save weights
Copy `best.pt` to Drive, then on your Mac: `python scripts/export.py best.pt` (INT8 Core ML needs macOS).

In [ ]:
!mkdir -p /content/drive/MyDrive/debris-data/weights && cp runs/*/weights/best.pt /content/drive/MyDrive/debris-data/weights/ && ls -la /content/drive/MyDrive/debris-data/weights